# DA-GPS vs OpenDSS daily compare — Google Colab (CUDA GPU)

Runs `nonunique.ipynb` cell 2 (`mode="da_gps_daily_compare"`): the DA-GPS GNN vs
OpenDSS native daily QSTS truth, on a Colab CUDA GPU.

**Before you start:** open `Runtime > Change runtime type > GPU` (T4 is fine).

Run the cells top to bottom. The GNN auto-detects CUDA and uses the GPU.

## What ships in git vs what you must upload
- **In git (cloned automatically):** all code modules, the grid DSS folder, the
  reference load/PV profiles, the edge CSV, and the small model checkpoint
  (`training_last.pt` + norm/sidecar tensors, ~9 MB).
- **NOT in git (too big for GitHub, you must provide it once):** the tensor cache
  `run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt`
  (~359 MB). Upload it to your Google Drive and paste its file ID below.

## 1. Clone the repo
If the GitHub repo is **private**, replace the URL with a token form, e.g.
`https://<USERNAME>:<PERSONAL_ACCESS_TOKEN>@github.com/alitasavori/GNN-Sandia.git`.

In [ ]:
REPO_URL = "https://github.com/alitasavori/GNN-Sandia.git"  # private? use a token URL
REPO_DIR = "/content/GNN2"

import os
if not os.path.isdir(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    print("Repo already cloned at", REPO_DIR)
%cd $REPO_DIR
!git log --oneline -3

## 2. Install dependencies
Colab already provides a CUDA-enabled `torch`. `torch_geometric` (>=2.4) needs no
external `torch-scatter`/`torch-sparse` for this model (only `GINEConv`/`Data`).

In [ ]:
!pip install -q torch_geometric "opendssdirect.py"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Portable paths + GPU env
`GNN2_REPO_ROOT` makes the modules resolve all data/checkpoint paths to the cloned
repo. `GNN_TORCH_COMPILE=0` skips `torch.compile` (faster startup on Colab).

In [ ]:
import os
os.environ["GNN2_REPO_ROOT"] = "/content/GNN2"
os.environ["GNN_TORCH_COMPILE"] = "0"

## 4. Fetch the large tensor cache (~359 MB) — one-time
This file is **not** in git. Upload it to Google Drive, make it shareable
("Anyone with the link"), copy the ID from the share URL
(`https://drive.google.com/file/d/<FILE_ID>/view`), and paste it below.

It is downloaded into the exact path the loader expects:
`datasets_gnn2_from pc/run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt`.

In [ ]:
# TODO(user): paste your Google Drive file ID for the cache .pt
CACHE_DRIVE_FILE_ID = "PASTE_DRIVE_FILE_ID_HERE"

import os
CACHE_REL = (
    "datasets_gnn2_from pc/"
    "run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt"
)
CACHE_PATH = os.path.join("/content/GNN2", CACHE_REL)
os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)

if os.path.isfile(CACHE_PATH) and os.path.getsize(CACHE_PATH) > 100_000_000:
    print("Cache already present:", CACHE_PATH)
else:
    if CACHE_DRIVE_FILE_ID == "PASTE_DRIVE_FILE_ID_HERE":
        raise SystemExit(
            "Set CACHE_DRIVE_FILE_ID to your Drive file ID first (see markdown above).\n"
            "Alternatively, mount Drive and copy the file to:\n  " + CACHE_PATH
        )
    !pip install -q gdown
    import gdown
    gdown.download(id=CACHE_DRIVE_FILE_ID, output=CACHE_PATH, quiet=False)
    print("Downloaded ->", CACHE_PATH, os.path.getsize(CACHE_PATH) / 1e6, "MB")

### (Alternative) Mount Google Drive instead of gdown
If you prefer mounting Drive, run this instead of the cell above and adjust the
source path to wherever you stored the file.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil, os
# SRC = '/content/drive/MyDrive/run_001_scen_0000_0049_seed_20420233__full__nobess__regce__mauxb7bd1d58.pt'
# os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
# shutil.copy(SRC, CACHE_PATH)
# print('copied ->', CACHE_PATH)

## 5. Run the compare (quick smoke first)
`npts=12` is a fast sanity check. Once it works, run the full-day cell below.

In [ ]:
%matplotlib inline
import sys
for _m in (
    "nonunique_daily_experiment", "nonunique_da_gps_daily_compare",
    "nonunique_warmstart_compare", "nonunique_four_scenario_demo",
    "nonunique_opendss_daily", "nonunique_da_gps", "nonunique_plots",
    "run_da_gps_daily_opendss_compare",
):
    sys.modules.pop(_m, None)
from nonunique_daily_experiment import run_and_plot

# Quick smoke: 12 points. DA-GPS auto-uses CUDA when available.
run_and_plot(
    mode="da_gps_daily_compare",
    step_min=5,
    npts=12,
    include_der=False,
    include_da_gps=True,
    show=True,
)

## 6. Full-day run (288 points @ 5 min)
Drop the `npts` override to run the full day, matching `nonunique.ipynb` cell 2.

In [ ]:
%matplotlib inline
import sys
for _m in (
    "nonunique_daily_experiment", "nonunique_da_gps_daily_compare",
    "nonunique_warmstart_compare", "nonunique_four_scenario_demo",
    "nonunique_opendss_daily", "nonunique_da_gps", "nonunique_plots",
    "run_da_gps_daily_opendss_compare",
):
    sys.modules.pop(_m, None)
from nonunique_daily_experiment import run_and_plot

run_and_plot(
    mode="da_gps_daily_compare",
    step_min=5,
    include_der=False,
    include_da_gps=True,
    show=True,
)